# Project K.A.R.E.N: AI-Enriched Threat Analysis 
This notebook demonstrates a **Hybrid AI Data Pipeline**. Rather than passing raw, unstructured datasets directly to a Large Language Model (which causes hallucinations and massive token costs), this pipeline relies on deterministic data engineering for heavy lifting, and reserves AI purely for synthesis.

### Pipeline Architecture:
1. **Deterministic Filtering (PySpark):** Ingest NYC Public School crime data, handle missing values, and calculate a custom `Spidey_Threat_Score` to isolate the top 3 high-risk zones. 
2. **Secure State Management:** Isolate data assets and API keys using **Databricks Unity Catalog**.
3. **Concurrent AI Extraction:** Dispatch 3 concurrent REST API calls to `gemini-3.8-flash` (with exponential backoff) to parse the numeric data into structured tactical JSON briefs.
4. **Geospatial UI:** Render the hotspots client-side using Folium/OpenStreetMap.
5. **AI Strategic Synthesis:** A final LLM pass evaluates the 3 briefs to issue a definitive deployment protocol based on civilian density and threat levels.

*(Total Pipeline LLM API Calls: Strictly capped at 4)*

## 1. Enterprise Setup & Unity Catalog Governance
To prevent polluting shared workspaces, we establish a dedicated schema (`workspace.spidey`). All resulting Delta tables and views are isolated here. 

By leveraging Unity Catalog, we ensure complete data lineage and establish a secure boundary for downstream BI tools or production jobs.


In [0]:
# Create the project schema in Unity Catalog if it does not already exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spidey")

# Set the current catalog and schema context
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA spidey")

DataFrame[]

## 2. Deterministic Filtering & Feature Engineering
**Goal:** Reduce millions of potential tokens down to only the exact context the LLM needs.

Using `PySpark`, we pull from the Databricks Marketplace, defensively cast nulls to `0` using `coalesce()` to protect downstream arithmetic, and engineer a custom `Spidey_Threat_Score`. We then filter the dataset down to the Top 3 targets and persist the results as a managed Delta Table.

In [0]:
from pyspark.sql.functions import col, desc, coalesce, lit

# 1. Load the raw source table from Databricks Marketplace
raw_crime_df = spark.table("us_crime_data.us_crime_data.crime_data_for_nyc_public_schools")

# 2. Clean nulls in numeric columns to ensure calculation reliability
cleaned_df = raw_crime_df.withColumn("Violent_Crime", coalesce(col("Violent_Crime"), lit(0))) \
                         .withColumn("Major_Crimes", coalesce(col("Major_Crimes"), lit(0))) \
                         .withColumn("Property_Crimes", coalesce(col("Property_Crimes"), lit(0))) \
                         .withColumn("Registered_Students", coalesce(col("Registered_Students"), lit(0)))

# 3. Feature Engineering: Spidey Threat Score
# Weigh violent offenses and major crimes highest, supplemented by property crime
scored_df = cleaned_df.withColumn(
    "Spidey_Threat_Score",
    (col("Violent_Crime") * 3) + (col("Major_Crimes") * 2) + col("Property_Crimes")
)

# 4. Filter, select essential attributes, and rank top 3
top_3_df = scored_df.select(
    "Location_Name",
    "School_Address",
    "Borough",
    "NTA",
    "Registered_Students",
    "Building_Population",
    "Violent_Crime",
    "Major_Crimes",
    "Property_Crimes",
    "Spidey_Threat_Score",
    "Latitude",
    "Longitude"
).orderBy(desc("Spidey_Threat_Score")).limit(3)

# 5. Persist as a Delta table in your dedicated schema for lineage and auditability
top_3_df.write.mode("overwrite").format("delta").saveAsTable("workspace.spidey.hotspot_targets")

# 6. Extract records as a Python list of dictionaries for the LLM step
top_3_schools = top_3_df.toPandas().to_dict(orient="records")

In [0]:
display(spark.table("workspace.spidey.hotspot_targets"))

Location_Name,School_Address,Borough,NTA,Registered_Students,Building_Population,Violent_Crime,Major_Crimes,Property_Crimes,Spidey_Threat_Score,Latitude,Longitude
800 EAST GUN HILL ROAD CONSOLIDATED LOCATION,800 EAST GUN HILL ROAD,X,Williamsbridge-Olinville,2816,2501-3000,20,6,11,83,40.875953,-73.86197
100 WEST MOSHOLU PARKWAY SOUTH CONSOLIDATED LOCATION,100 WEST MOSHOLU PARKWAY SOUTH,X,Van Cortlandt Village,2905,2501-3000,17,8,7,74,40.882178,-73.88691
99 TERRACE VIEW AVENUE CONSOLIDATED LOCATION,99 TERRACE VIEW AVENUE,X,Marble Hill-Inwood,2517,2501-3000,7,15,22,73,40.877105,-73.912256


## 3. Concurrent AI Extraction (LLM Calls 1-3)
Calling APIs in a standard `for` loop creates I/O bottlenecks. Instead, we use Python's `concurrent.futures.ThreadPoolExecutor` to hit the Gemini REST endpoint concurrently.

**Production Engineering Features:**
* **Native REST:** Using `requests` avoids pip dependency conflicts on the cluster.
* **Security:** API keys are injected securely via `dbutils.secrets.get(catalog, schema)`.
* **Resilience:** Built-in exponential backoff with jitter to gracefully handle `429` (Rate Limit) and `503` (Service Unavailable) errors.
* **Structured Output:** Model configuration is locked to `application/json` to ensure a strict contract for downstream parsing.

In [0]:
import json
import time
import random
import requests
import concurrent.futures

# --- Configuration ---
MODEL = "gemini-3.8-flash"
API_KEY = dbutils.secrets.get(catalog="workspace", schema="spidey", key="gemini_api_key")
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

MAX_ATTEMPTS = 4
REQUEST_TIMEOUT = 60
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

def get_situation_brief(school):
    """
    Calls Gemini REST endpoint to extract a structured situation brief.
    Uses exponential backoff + jitter for retryable HTTP errors.
    """
    data_context = json.dumps(school)
    
    prompt = f"""
    You are the 'Karen' AI suite in Peter Parker's suit. Analyze this school location and 
    output a short, punchy tactical brief. 
    
    Data: {data_context}
    
    Format as JSON:
    {{
        "School": "Name",
        "Threat_Level": "1-10 scale based on violent/major crimes",
        "Civilian_Risk": "High/Med/Low based on Registered_Students count",
        "Karen_Analysis": "2 sentences on what Spidey should expect"
    }}
    """

    payload = {
        "contents": [{
            "parts": [{"text": prompt}]
        }],
        "generationConfig": {
            "responseMimeType": "application/json",
            "temperature": 0.2
        }
    }

    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": API_KEY
    }

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.post(
                ENDPOINT,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT
            )

            if response.status_code == 200:
                body = response.json()
                raw_text = body["candidates"][0]["content"]["parts"][0]["text"]
                return json.loads(raw_text)

            elif response.status_code in RETRYABLE_STATUS_CODES:
                if attempt == MAX_ATTEMPTS:
                    raise RuntimeError(f"Exhausted retries ({response.status_code}): {response.text}")
                sleep_time = (2 ** attempt) + random.uniform(0.1, 0.8)
                time.sleep(sleep_time)

            else:
                raise RuntimeError(f"Non-retryable HTTP {response.status_code}: {response.text}")

        except requests.RequestException as e:
            if attempt == MAX_ATTEMPTS:
                raise RuntimeError(f"Request failed after {MAX_ATTEMPTS} attempts: {e}")
            time.sleep((2 ** attempt) + random.uniform(0.1, 0.8))

# Concurrently dispatch the 3 calls
print("Connecting to Karen AI interface...")
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    karen_briefs = list(executor.map(get_situation_brief, top_3_schools))

print(json.dumps(karen_briefs, indent=2))

Connecting to Karen AI interface...
[
  {
    "School": "800 East Gun Hill Road Consolidated Location",
    "Threat_Level": "8/10",
    "Civilian_Risk": "High",
    "Karen_Analysis": "Peter, with over 2,800 students on-site and elevated violent crime stats in the perimeter, this is a volatile hotspot. Stick to the high rafters for surveillance and keep your web-shooters calibrated for non-lethal crowd suppression."
  },
  {
    "School": "100 West Mosholu Parkway South Consolidated Location",
    "Threat_Level": "7/10",
    "Civilian_Risk": "High",
    "Karen_Analysis": "Peter, this campus houses nearly 3,000 students alongside an elevated record of 17 violent incidents, making high civilian density your biggest tactical obstacle. I recommend prioritizing non-lethal crowd-control measures and monitoring the perimeter exits closely."
  },
  {
    "School": "99 Terrace View Avenue Consolidated Location",
    "Threat_Level": "7/10",
    "Civilian_Risk": "High",
    "Karen_Analysis": "Pete

## 4. Geospatial UI & Situational Awareness
Before finalizing the deployment, we render the target coordinates. This utilizes Folium with an `OpenStreetMap` tileset (bypassing CartoDB API key requirements) to create a zero-dependency, interactive HTML map directly inside the notebook environment.

In [0]:
%pip install -q folium

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import folium
from folium.plugins import MarkerCluster

# Explicitly defining the OSM attribution to comply with ODbL safe harbour guidelines
osm_attribution = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'

# Initialize the map with the explicit attribution
m = folium.Map(
    location=[40.7128, -74.0060], 
    zoom_start=10, 
    tiles="OpenStreetMap",
    attr=osm_attribution
)

# 2. Iterate through the top 3 schools to add markers
for school in top_3_schools:
    lat = school.get("Latitude")
    lon = school.get("Longitude")
    
    # Ensure coordinates exist before plotting
    if lat and lon:
        # Create a visually striking HTML popup for the map
        popup_html = f"""
        <div style="font-family: Arial; width: 200px;">
            <h4 style="color: #ff0000; margin-bottom: 5px;">THREAT LEVEL: {school.get('Spidey_Threat_Score')}</h4>
            <b>{school.get('Location_Name')}</b><br>
            <i>{school.get('Borough')}</i><br>
            <hr style="margin: 5px 0;">
            <b>Violent Crimes:</b> {school.get('Violent_Crime')}<br>
            <b>Major Crimes:</b> {school.get('Major_Crimes')}<br>
            <b>Student Density:</b> {school.get('Registered_Students')}
        </div>
        """
        
        # Add a custom red marker for the hotspots
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(popup_html, max_width=250),
            icon=folium.Icon(color="red", icon="info-sign"),
            tooltip="Click for Karen AI Threat Data"
        ).add_to(m)

# 3. Automatically zoom the map to fit the 3 pins perfectly
sw = [min([s['Latitude'] for s in top_3_schools if s['Latitude']]), 
      min([s['Longitude'] for s in top_3_schools if s['Longitude']])]
ne = [max([s['Latitude'] for s in top_3_schools if s['Latitude']]), 
      max([s['Longitude'] for s in top_3_schools if s['Longitude']])]
m.fit_bounds([sw, ne])

# 4. Render the interactive map in the Databricks notebook
displayHTML(m._repr_html_())

Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_1f222badd29278fafb6867dc9e77289d {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>

 <style>html, body {
 width: 100%;
 height: 100%;
 margin: 0;
 padding: 0;
 }
 </style>

 <style>#map {
 position:absolute;
 top:0;
 bottom:0;
 right:0;
 left:0;
 }
 </style>

 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>

 
</head>
<body>
 
 
 <div class="folium-map" id="map_1f222badd29278fafb6867dc9e77289d" ></div>
 
</body>
<script>
 
 
 var map_1f222badd29278fafb6867dc9e77289d = L.map(
 "map_1f222badd29278fafb6867dc9e77289d",
 {
 center: [40.7128, -74.006],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 10,
 "zoomControl": true,
 "preferCanvas": false,
}

 }
 );

 

 
 
 var tile_layer_8d7d5d4e1b14a70d36daca633ee30f99 = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_8d7d5d4e1b14a70d36daca633ee30f99.addTo(map_1f222badd29278fafb6867dc9e77289d);
 
 
 var marker_8a17840df2198055aa587194163db350 = L.marker(
 [40.875953, -73.86197],
 {
}
 ).addTo(map_1f222badd29278fafb6867dc9e77289d);
 
 
 var icon_a57f772942f1f69c1a4af3ebb884b0d7 = L.AwesomeMarkers.icon(
 {
 "markerColor": "red",
 "iconColor": "white",
 "icon": "info-sign",
 "prefix": "glyphicon",
 "extraClasses": "fa-rotate-0",
}
 );
 
 
 var popup_499638df148f886dd7dca2def19a195d = L.popup({
 "maxWidth": 250,
});

 
 
 var html_a16e8d3205b988ed01e5a1f19d0abd21 = $(`<div id="html_a16e8d3205b988ed01e5a1f19d0abd21" style="width: 100.0%; height: 100.0%;"> <div style="font-family: Arial; width: 200px;"> <h4 style="color: #ff0000; margin-bottom: 5px;">THREAT LEVEL: 83</h4> <b>800 EAST GUN HILL ROAD CONSOLIDATED LOCATION</b><br> <i>X</i><br> <hr style="margin: 5px 0;"> <b>Violent Crimes:</b> 20<br> <b>Major Crimes:</b> 6<br> <b>Student Density:</b> 2816 </div> </div>`)[0];
 popup_499638df148f886dd7dca2def19a195d.setContent(html_a16e8d3205b988ed01e5a1f19d0abd21);
 
 

 marker_8a17840df2198055aa587194163db350.bindPopup(popup_499638df148f886dd7dca2def19a195d)
 ;

 
 
 
 marker_8a17840df2198055aa587194163db350.bindTooltip(
 `<div>
 Click for Karen AI Threat Data
 </div>`,
 {
 "sticky": true,
}
 );
 
 
 marker_8a17840df2198055aa587194163db350.setIcon(icon_a57f772942f1f69c1a4af3ebb884b0d7);
 
 
 var marker_1ced9675fc0c30d7985889a7c5b5865d = L.mark

## 5. Strategic AI Synthesis (LLM Call 4)
With the raw data successfully translated into 3 localized JSON briefs, we invoke the LLM one final time. The prompt instructs the model to weigh the *Civilian Risk* (student density) against the *Threat Level* to determine the optimal deployment target. 

By separating the **extraction** step (Call 1-3) from the **reasoning** step (Call 4), we achieve highly predictable, interpretable, and token-efficient AI workflows.

In [0]:
from IPython.display import Markdown

def deploy_spiderman(briefs):
    prompt = f"""
    You are Spider-Man's tactical AI. Review the following 3 situation briefs for 
    NYC public schools: {json.dumps(briefs)}
    
    Determine which school Spider-Man should swing to FIRST right now. 
    Write a 2-paragraph rationale explaining your choice to Peter. 
    Speak directly to him. Weigh civilian risk (student population) against the threat level.
    """

    payload = {
        "contents": [{
            "parts": [{"text": prompt}]
        }],
        "generationConfig": {
            "temperature": 0.7
        }
    }

    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": API_KEY
    }

    response = requests.post(
        ENDPOINT,
        headers=headers,
        json=payload,
        timeout=REQUEST_TIMEOUT
    )
    
    if response.status_code == 200:
        return response.json()["candidates"][0]["content"]["parts"][0]["text"]
    else:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text}")

final_deployment_plan = deploy_spiderman(karen_briefs)
display(Markdown(f"### Spider-Man Deployment Protocol\n\n{final_deployment_plan}"))

### Spider-Man Deployment Protocol

Peter, you need to reroute immediately toward **800 East Gun Hill Road**. While all three locations present critical civilian exposure, Gun Hill Road is registering the highest active Threat Level at an 8/10. With over 2,800 students concentrated on campus and a surge of violent crime along the immediate perimeter, the probability of an active escalation spilling into the student body is significantly higher here than anywhere else on your HUD. An 8/10 threat rating in an enclosed educational facility means hostiles are likely already armed, organized, or actively breaching protocols, making every second count before the bell rings and corridors fill.

I ran a risk-multiplication matrix against the other two campuses to be certain. While 100 West Mosholu Parkway South edges out Gun Hill in raw population by roughly 200 students, its lower threat rating of 7/10 indicates a comparatively more stable perimeter, as does 99 Terrace View Avenue. We cannot trade off an active, higher-severity threat for a marginal 7% increase in civilian density. Head to Gun Hill Road first, stick to the upper rafters to maintain line-of-sight over the courtyard, and set your web-shooters to wide-spread non-lethal dispersal—I’ll monitor emergency frequencies at Mosholu while you secure the perimeter.